In [54]:



from pptx import Presentation
from pptx.util import Inches, Pt
import pandas as pd
from pathlib import Path

In [91]:

def create_pptx( res_filename, mah_gifs, perc_gifs, dists, percs,sutures, comments, title):
    # Create presentation
    prs = Presentation()
    blank_slide_layout = prs.slide_layouts[6]  # Blank slide layout

    print("lenght of slides", len(mah_gifs))
    for i in range(len(mah_gifs)):


    # Add a slide
        slide = prs.slides.add_slide(blank_slide_layout)

        # Add title
        title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(5), Inches(1))
        title_tf = title_box.text_frame
        print(title)
        title_tf.text = title
        title_tf.paragraphs[0].font.size = Pt(44)

        # Add GIF 1 (left)
        slide.shapes.add_picture(mah_gifs[i], Inches(0.5), Inches(1.5), width=Inches(3.5))

        # Add GIF 2 (right)
        slide.shapes.add_picture(perc_gifs[i], Inches(4.5), Inches(1.5), width=Inches(3.5))

        # Add metrics textbox
        metrics_box = slide.shapes.add_textbox(Inches(0.5), Inches(5.3), Inches(3), Inches(1))
        metrics_tf = metrics_box.text_frame
        metrics_tf.text = f"Dist {dists[i]} perc\n{percs[i]} \nSutures: {sutures[i]}"
        for p in metrics_tf.paragraphs:
            p.font.size = Pt(14)

        # Add comment box
        comment_box = slide.shapes.add_textbox(Inches(3.6), Inches(4.3), Inches(3), Inches(1))
        comment_tf = comment_box.text_frame
        print(comments[i])

        comm_str = ""
        for k,v in comments[i].items():
            comm_str += f"{k}: {v}\n"
        comment_tf.text = comm_str

        # Set font size for all lines
        for paragraph in comment_tf.paragraphs:
            paragraph.font.size = Pt(12)


    # Save the presentation
    prs.save(res_filename)





In [81]:
genes_csv = '/Users/igor.varha/Downloads/Single mouse values-1.csv'

all_cohorts_folder = '/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp'

res_paths = '/Users/igor.varha/Downloads/pptx'
cohor_dict = { 'c' + x.name.split('_')[0].split('c')[-1]: x for x in Path(all_cohorts_folder).iterdir()if x.is_dir() }

genes = pd.read_csv(genes_csv,index_col=0)

In [29]:
df = genes[genes["COHORT_ID"] >"c200"]
# group by gene
gr = df.groupby("GENE")
df.columns

Index(['GENDER', 'COHORT_ID', 'SPECIMEN_ID', 'GENE', 'Comment on XRay image',
       'Length of cranium', 'Mandibles', 'Maxilla/Pre-maxilla', 'Skull shape',
       'Zygomatic bone'],
      dtype='object')

In [21]:
gr.groups.keys()

dict_keys(['Bmp6', 'Col2a1', 'Col5a2', 'Cst6', 'Kif6', 'Meis3', 'Mrvi1', 'Napa', 'Nipal2', 'Pgf', 'Snurf', 'Ttll13'])

In [93]:
for gene, group in gr:

    mah_gifs = []
    perc_gifs = []
    dists = []
    percs = []
    sutures = []

    all_info = []



    fields = ['GENDER', 'COHORT_ID','SPECIMEN_ID', 'GENE', 'Comment on XRay image',
       'Length of cranium', 'Mandibles', 'Maxilla/Pre-maxilla', 'Skull shape',
       'Zygomatic bone']

    # iterate over group

    for index, row in group.iterrows():

        # get the cohort
        cohort = row["COHORT_ID"]

        if cohort not in cohor_dict:
            print(f"Skipping {cohort} due to missing folder")
            continue

        if not (cohor_dict[cohort]/'skull_shape_workdir').exists():
            print(f"Skipping {cohort} due to missing skull_shape_workdir")
            continue

        # speciment id
        specimen_id = row["SPECIMEN_ID"]

        # get skull shape folder (ends with specimen_id)
        skull_shape_folder = [Path(x) for x in (cohor_dict[cohort]/'skull_shape_workdir').iterdir() if x.name.endswith(specimen_id)][0]

        if not ((skull_shape_folder/"mahalanobis.gif").exists()):
            print(f"Skipping {cohort} due to missing mahalanobis.gif")
            continue
        if not ((skull_shape_folder/"perc.gif").exists()):
            print(f"Skipping {cohort} due to missing percentiles.gif")
            continue


        if not ((skull_shape_folder/"abnormality_automated.csv").exists()):
            print(f"Skipping {cohort} due to missing abnormality_automated.csv")
            continue

        # read abnormality_automated.csv
        abnormality = pd.read_csv(skull_shape_folder/"abnormality_automated.csv",index_col=0).transpose().iloc[0]

        proc_dis = abnormality["proc_dist"]
        area99 = abnormality["area99perc"]

        if 'sutures_open' in abnormality.index:
            sutures_open = abnormality['sutures_open']
        else:
            sutures_open = None


        mah_gifs.append(str(skull_shape_folder/"mahalanobis.gif"))
        perc_gifs.append(str(skull_shape_folder/"perc.gif"))
        dists.append(proc_dis)
        percs.append(area99)
        sutures.append(sutures_open)

        all_info.append(row[fields].to_dict())
    res_filename = Path(res_paths)/f"{gene}.pptx"

    create_pptx(res_filename, mah_gifs, perc_gifs, dists, percs, sutures,all_info, title=gene)

    # if len(mah_gifs) > 0:
    #     break











        #print(skull_shape_folder)
        #print(row)



Skipping c205 due to missing folder
lenght of slides 0
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
Skipping c364 due to missing skull_shape_workdir
lenght of slides 0
lenght of slides 16
Col5a2
{'GENDER': 'female', 'COHORT_ID': 'c347', 'SPECIMEN_ID': 'E64-CJ736', 'GENE': 'Col5a2', 'Comment on XRay image': 'T

In [26]:
cohor_dict

{'c231': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20220321c231_Ccdc198 & Myo1g & Rras2 & wt & Zc3h8'),
 'c349': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20240729c349_Clip2 & Gadl1 & Gfi1b & Gtf2ird1 & wt'),
 'c290': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20230529c290_Adam7 & Cd209c & wt'),
 'c324': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20240122c324_Fgf23 & wt & Zim1'),
 'c292': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20230612c292_Gcnt1 & Pcid2 & Tcirg1 & wt'),
 'c245': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20220704c245_Krt1 & Nfe2l2 & Tcaf2 & wt'),
 'c326': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20240205c326_Galnt3KO & Sval1 & wt & Zcrb1'),
 'c253': PosixPath('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.

In [71]:
comments

NameError: name 'comments' is not defined

In [89]:
pd.read_csv('/Volumes/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20220905c254_Dcaf15 & Pgc & wt/skull_shape_workdir/Dcaf15_F_E64-G8137/abnormality_automated.csv',index_col=0).transpose().iloc[0]

proc_dist          629.4381235511187
abn_distance                  normal
area99perc      0.029245609315552822
abn_perc                      normal
s_x               1.0153938976406185
s_y                0.987970587259736
s_z               0.9771044308393676
Name: 0, dtype: object

In [76]:
all_info[-1]

{'GENDER': 'male',
 'SPECIMEN_ID': 'E64-CJ760',
 'GENE': 'Col5a2',
 'Comment on XRay image': 'T1 not fused, pectus excavatum mild',
 'Length of cranium': 26.43,
 'Mandibles': 'normal',
 'Maxilla/Pre-maxilla': 'normal',
 'Skull shape': 'normal',
 'Zygomatic bone': 'normal'}